# אימון YOLOv8 על המשחק (ex4_demo) — Google Colab

לפני שמריצים: **Runtime → Change runtime type → T4 GPU**.

מריצים את התאים לפי הסדר. בתא 2 מעלים את `backend/train/dataset.zip` מהמחשב.

In [ ]:
!pip -q install ultralytics
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none — switch runtime to GPU')

In [ ]:
# 2. upload dataset.zip (backend/train/dataset.zip)
from google.colab import files
up = files.upload()
!rm -rf /content/dataset && unzip -q dataset.zip -d /content
# data.yaml holds a Windows path — point it at Colab
open('/content/dataset/data.yaml','w').write('''path: /content/dataset
train: images/train
val: images/val
names:
  0: character
  1: tree
  2: fruit
  3: sun
''')
!ls /content/dataset/images/train | wc -l; ls /content/dataset/images/val | wc -l

In [ ]:
# 3. train (~5-10 min on a T4)
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
model.train(data='/content/dataset/data.yaml', epochs=40, imgsz=800, batch=16, device=0,
            project='/content/runs', name='game', exist_ok=True, seed=0)

In [ ]:
# 4. metrics per class on the validation split
m = YOLO('/content/runs/game/weights/best.pt')
r = m.val(data='/content/dataset/data.yaml', imgsz=800, plots=False)
for i, n in r.names.items():
    print(f'{n:10s} mAP50={r.box.ap50[i]:.3f}  mAP50-95={r.box.maps[i]:.3f}')
print('all        mAP50=%.3f' % r.box.map50)

In [ ]:
# 5. quick look: predictions on a few validation frames
import glob, cv2
from IPython.display import Image, display
for p in sorted(glob.glob('/content/dataset/images/val/*.jpg'))[:3]:
    res = m.predict(p, imgsz=800, conf=0.3, verbose=False)[0]
    cv2.imwrite('/content/pred.jpg', cv2.resize(res.plot(), (960, 540)))
    display(Image('/content/pred.jpg'))

In [ ]:
# 6. download best.pt  ->  save it as backend/game.pt
files.download('/content/runs/game/weights/best.pt')

## חזרה למחשב

```
copy best.pt backend\game.pt
set YOLO_MODEL=game.pt
set IMGSZ=800
uvicorn main:app --port 8000
```